## Когортный анализ retention + визуализации

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
import os

# ====================== НАСТРОЙКИ ======================
DB_USER = 'root'          # измени, если нужно
DB_PASSWORD = ''          # укажи свой пароль или оставь пустым
DB_HOST = 'localhost'
DB_NAME = 'food_delivery'

engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}')

print("✅ Подключение к MySQL установлено")

✅ Подключение к MySQL установлено


In [2]:
# ====================== НАСТРОЙКИ ПОДКЛЮЧЕНИЯ ======================
DB_USER = 'root'
DB_PASSWORD = '213360AzQh!'          # ←←← ВВЕДИ СВОЙ ПАРОЛЬ ЗДЕСЬ (если пароля нет — оставь '')
DB_HOST = 'localhost'
DB_NAME = 'food_delivery'

# Если пароль пустой, можно использовать такой вариант:
if not DB_PASSWORD:
    engine = create_engine(f'mysql+pymysql://{DB_USER}@{DB_HOST}/{DB_NAME}')
else:
    engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}')

print("✅ Подключение к MySQL настроено")
print(f"Пользователь: {DB_USER}, База: {DB_NAME}")

✅ Подключение к MySQL настроено
Пользователь: root, База: food_delivery


In [3]:
# Загружаем данные когортного retention
query = """
WITH user_cohorts AS (
    SELECT
        u.user_id,
        u.ab_group,
        u.device,
        u.acquisition_channel,
        u.city,
        DATE(u.registration_date) AS reg_date,
        DATE_SUB(DATE(u.registration_date), INTERVAL WEEKDAY(u.registration_date) DAY) AS cohort_week
    FROM users u
),

first_activity AS (
    SELECT
        uc.user_id,
        uc.cohort_week,
        uc.ab_group,
        uc.device,
        uc.acquisition_channel,
        uc.city,
        uc.reg_date,
        MIN(DATE(o.order_date)) AS first_order_date
    FROM user_cohorts uc
    LEFT JOIN orders o ON uc.user_id = o.user_id
    GROUP BY uc.user_id, uc.cohort_week, uc.ab_group,
             uc.device, uc.acquisition_channel, uc.city, uc.reg_date
)

SELECT
    cohort_week,
    ab_group,
    device,
    acquisition_channel,
    city,
    COUNT(DISTINCT user_id) AS cohort_size,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_date, reg_date) = 0 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_d0_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_date, reg_date) <= 1 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_d1_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_date, reg_date) <= 7 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_d7_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_date, reg_date) <= 14 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_d14_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_date, reg_date) <= 30 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_d30_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN TIMESTAMPDIFF(WEEK, reg_date, first_order_date) = 0 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_w0_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN TIMESTAMPDIFF(WEEK, reg_date, first_order_date) <= 1 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_w1_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN TIMESTAMPDIFF(WEEK, reg_date, first_order_date) <= 4 THEN user_id END)
          / COUNT(DISTINCT user_id), 2) AS ret_w4_pct
FROM first_activity
WHERE first_order_date IS NOT NULL
GROUP BY cohort_week, ab_group, device, acquisition_channel, city
HAVING cohort_size >= 300
ORDER BY cohort_week, ab_group, device, acquisition_channel, city;
"""

df = pd.read_sql(query, engine)
print(f"✅ Загружено {len(df):,} строк когорт")
df.head()

✅ Загружено 28 строк когорт


,cohort_week,ab_group,device,acquisition_channel,city,cohort_size,ret_d0_pct,ret_d1_pct,ret_d7_pct,ret_d14_pct,ret_d30_pct,ret_w0_pct,ret_w1_pct,ret_w4_pct
0,2024-12-30,A,android,google_ads,Москва,382,2.62,5.76,19.37,33.77,57.59,17.02,31.68,62.83
1,2024-12-30,A,android,organic,Москва,503,3.18,5.57,22.07,36.78,61.63,20.28,35.39,66.40
2,2024-12-30,A,android,organic,Санкт-Петербург,314,3.50,4.46,20.38,33.44,55.41,18.47,31.21,61.78
3,2024-12-30,A,android,yandex_direct,Москва,320,3.44,6.25,20.94,36.88,62.19,18.75,34.38,68.13
4,2024-12-30,A,ios,organic,Москва,318,3.46,5.97,19.81,34.28,61.32,17.92,32.39,66.67


In [4]:
# Heatmap retention (D1 / D7 / D14 / D30)
pivot = df.pivot_table(
    index='cohort_week',
    columns='ab_group',
    values='ret_d7_pct',
    aggfunc='mean'
).round(1)

fig_heatmap = px.imshow(
    pivot,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="Retention D7 по когортам и AB-группам (%)"
)
fig_heatmap.update_layout(height=600)
fig_heatmap.show()

# Сохраняем
os.makedirs('../images', exist_ok=True)
fig_heatmap.write_image("../images/retention_heatmap_d7.png")

In [5]:
# Retention curves
fig_curves = go.Figure()

for group in ['A', 'B']:
    data = df[df['ab_group'] == group].groupby('cohort_week').mean(numeric_only=True)
    fig_curves.add_trace(go.Scatter(
        x=data.index,
        y=data['ret_d1_pct'],
        mode='lines+markers',
        name=f'Group {group} — D1',
        line=dict(width=3)
    ))
    fig_curves.add_trace(go.Scatter(
        x=data.index,
        y=data['ret_d7_pct'],
        mode='lines+markers',
        name=f'Group {group} — D7',
        line=dict(dash='dash')
    ))

fig_curves.update_layout(
    title="Retention Curves по AB-группам (D1 и D7)",
    xaxis_title="Когорта (неделя)",
    yaxis_title="Retention (%)",
    height=600,
    template="plotly_white"
)
fig_curves.show()

fig_curves.write_image("../images/retention_curves_ab.png")

- Retention D7 в группе B, чем в A →
- Самая сильная когорта — …
- Рекомендация: …